# Optimizing Dividend Capture Strategies Using Genetic Algorithms (Numba-JIT Accelerated)

This notebook optimizes entry and exit dividend capture trading signals using **Genetic Algorithms (GA)**.

### 🚀 Numba-JIT Performance Acceleration
To eliminate CPU overhead from native Python matrix/DataFrame iterations during generation passes, fitness calculations have been compiled into machine code using **Numba JIT** (`@njit(fastmath=True, parallel=True)`). Raw NumPy arrays are passed directly to compiled C-speed routines.

In [1]:
import pandas as pd
import numpy as np
import datetime

# Generate synthetic data for dividend capture strategies
np.random.seed(42)
dates = pd.date_range(start='2020-01-01', end='2023-12-31', freq='D')
stock_prices = np.random.uniform(50, 150, len(dates))
dividends = np.zeros(len(dates))
dividend_dates = np.random.choice(dates, size=30, replace=False)
dividends[[dates.get_loc(date) for date in dividend_dates]] = np.random.uniform(0.5, 2.0, len(dividend_dates))

# Create DataFrame
data_dividend = pd.DataFrame({
    'Date': dates,
    'Stock_Price': stock_prices,
    'Dividend': dividends
})

# Save to CSV
data_dividend.to_csv('dividend_data.csv', index=False)


In [2]:
import pandas as pd
import numpy as np

# Load data
data_dividend = pd.read_csv('dividend_data.csv')
print(data_dividend.head())


         Date  Stock_Price  Dividend
0  2020-01-01    87.454012       0.0
1  2020-01-02   145.071431       0.0
2  2020-01-03   123.199394       0.0
3  2020-01-04   109.865848       0.0
4  2020-01-05    65.601864       0.0


In [3]:
# Pure Python fitness function (Legacy / Baseline)
def fitness_function_py(individual, data):
    buy_signal = individual[0]
    sell_signal = individual[1]

    capital = 100000.0  # Starting capital
    position = 0.0
    returns = 0.0

    for i in range(len(data)):
        if data['Dividend'][i] > buy_signal and position == 0:
            position = capital / data['Stock_Price'][i]
            capital = 0.0

        if data['Dividend'][i] < sell_signal and position > 0:
            capital = position * data['Stock_Price'][i]
            returns += capital - 100000.0  # Net returns
            position = 0.0

    return returns


In [4]:
from numba import njit, prange

# Numba-JIT compiled fitness evaluation accepting raw NumPy arrays
@njit(fastmath=True)
def fitness_function_numba(individual, dividends, stock_prices):
    buy_signal = individual[0]
    sell_signal = individual[1]
    capital = 100000.0
    position = 0.0
    returns = 0.0
    n = len(dividends)

    for i in range(n):
        if dividends[i] > buy_signal and position == 0.0:
            position = capital / stock_prices[i]
            capital = 0.0

        if dividends[i] < sell_signal and position > 0.0:
            capital = position * stock_prices[i]
            returns += capital - 100000.0
            position = 0.0

    return returns

# Parallel population evaluation
@njit(fastmath=True, parallel=True)
def evaluate_population_numba(population, dividends, stock_prices):
    pop_size = population.shape[0]
    fitness_scores = np.empty(pop_size, dtype=np.float64)
    for p in prange(pop_size):
        fitness_scores[p] = fitness_function_numba(population[p], dividends, stock_prices)
    return fitness_scores


In [5]:
def initialize_population(pop_size):
    population = []
    for _ in range(pop_size):
        individual = np.random.uniform(0.01, 2.0, 2)  # Random buy and sell signals
        population.append(individual)
    return population

def selection(population, fitness_scores, num_parents):
    parents = [population[idx] for idx in np.argsort(fitness_scores)[-num_parents:]]
    return parents

def crossover(parents, offspring_size):
    offspring = []
    for _ in range(offspring_size):
        parent1 = parents[np.random.randint(len(parents))]
        parent2 = parents[np.random.randint(len(parents))]
        crossover_point = np.random.randint(1, len(parent1))
        child = np.concatenate((parent1[:crossover_point], parent2[crossover_point:]))
        offspring.append(child)
    return offspring

def mutation(offspring, mutation_rate):
    for individual in offspring:
        if np.random.rand() < mutation_rate:
            mutation_point = np.random.randint(len(individual))
            individual[mutation_point] = np.random.uniform(0.01, 2.0)
    return offspring


In [6]:
import time

# Benchmark 100-generation simulation pass: Native Python vs. Numba JIT
dividends_arr = data_dividend['Dividend'].to_numpy(dtype=np.float64)
prices_arr = data_dividend['Stock_Price'].to_numpy(dtype=np.float64)
sample_pop = np.array(initialize_population(100))

# Warmup Numba JIT compiler
_ = evaluate_population_numba(sample_pop, dividends_arr, prices_arr)

# 1. Native Python timing
t0 = time.perf_counter()
for _ in range(100):
    py_scores = [fitness_function_py(ind, data_dividend) for ind in sample_pop]
t_py_ms = (time.perf_counter() - t0) * 1000.0

# 2. Numba JIT Accelerated timing
t0 = time.perf_counter()
for _ in range(100):
    nb_scores = evaluate_population_numba(sample_pop, dividends_arr, prices_arr)
t_nb_ms = (time.perf_counter() - t0) * 1000.0

print(f"=== 100-Generation Simulation Benchmark ===")
print(f"Native Python Execution Time : {t_py_ms:.2f} ms")
print(f"Numba JIT Execution Time    : {t_nb_ms:.2f} ms")
print(f"Speedup Acceleration Factor : {t_py_ms / t_nb_ms:.1f}x faster")
print(f"Numerical Parity Check      : {np.allclose(py_scores, nb_scores)}")


=== 100-Generation Simulation Benchmark ===
Native Python Execution Time : 243743.20 ms
Numba JIT Execution Time    : 6.32 ms
Speedup Acceleration Factor : 38550.5x faster
Numerical Parity Check      : True


In [7]:
def genetic_algorithm_numba(data, num_generations, pop_size, num_parents, mutation_rate):
    dividends_arr = data['Dividend'].to_numpy(dtype=np.float64)
    prices_arr = data['Stock_Price'].to_numpy(dtype=np.float64)
    population = initialize_population(pop_size)

    for generation in range(num_generations):
        pop_mat = np.array(population)
        fitness_scores = evaluate_population_numba(pop_mat, dividends_arr, prices_arr)
        parents = selection(population, fitness_scores, num_parents)
        offspring_size = pop_size - len(parents)
        offspring = crossover(parents, offspring_size)
        offspring = mutation(offspring, mutation_rate)
        population = parents + offspring

        best_fitness = np.max(fitness_scores)
        if (generation + 1) % 10 == 0 or generation == 0:
            print(f"Generation {generation}: Best Fitness = {best_fitness:.2f}")

    pop_mat = np.array(population)
    final_scores = evaluate_population_numba(pop_mat, dividends_arr, prices_arr)
    best_individual = population[np.argmax(final_scores)]
    return best_individual

# Run the Numba-accelerated genetic algorithm
num_generations = 50
pop_size = 100
num_parents = 20
mutation_rate = 0.01

best_params = genetic_algorithm_numba(data_dividend, num_generations, pop_size, num_parents, mutation_rate)
print(f"\nBest Parameters (Buy Signal, Sell Signal): {best_params}")


Generation 0: Best Fitness = 1340050.11
Generation 9: Best Fitness = 1340050.11
Generation 19: Best Fitness = 1340050.11
Generation 29: Best Fitness = 1346751.02
Generation 39: Best Fitness = 1346751.02
Generation 49: Best Fitness = 1346751.02

Best Parameters (Buy Signal, Sell Signal): [0.53198681 1.60900104]
